# Training a phase reconstructor for Rama

This notebook trains a convolutional neural network to reconstruct wavefronts from Rama's pyramid WFS detector frames, using the fully differentiable end-to-end simulation.

Unlike the calibration notebook — where we fit the WFS and DM to match real bench measurements — here the WFS and DM are loaded from a previous calibration and then frozen (`requires_grad_(False)`); only the reconstructor network's weights are updated.

Training simulates an AO closed loop: at each step we draw a random turbulent phase screen, propagate the *residual* phase (after applying the previous correction) through the WFS, and backpropagate the reconstruction error through the optical model and DM straight into the network. This is implemented by `AI4AO.Trainer.Trainer` (see `AI4AO/Trainer.py`), which this notebook uses for training, evaluation, loss plotting, and checkpointing — replacing the hand-written closed-loop training loop this notebook used to have.

Rama's pyramid is also configured as an *elongated* (prism) pyramid rather than the standard 4-facet one; see the "Elongated pyramid mask" section below for what that changes and a bug it fixes.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss

## Reconstructor architecture

`PWFSNet` maps the 4 preprocessed pyramid pupil images to a vector of `Nmodes` mode coefficients: a convolutional encoder (two conv layers per resolution stage, deeper than the equivalent network used for Ekarus) followed by a linear head.

In [ ]:
class PWFSNet(nn.Module):
    def __init__(self, DMParams):
        super().__init__()

        Nmodes = DMParams["Nmodes"]

        self.stem = nn.Sequential(
            # Process each pupil independently
            nn.Conv2d(4, 32, kernel_size=11, padding=5, groups=4),
            nn.GELU(),

            nn.Conv2d(32, 64, kernel_size=7, padding=3, groups=4),
            nn.GELU(),

            nn.MaxPool2d(2),      # 42 -> 21
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(64, 64, 5, padding=2),
            nn.GELU(),

            nn.Conv2d(64, 64, 5, padding=2),
            nn.GELU(),

            nn.MaxPool2d(2),      # 21 -> 10

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GELU(),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),      # 10 -> 5

            nn.Conv2d(128, 256, 3, padding=1),
            nn.GELU(),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),      # 10 -> 5

            nn.Conv2d(256, 512, 2, padding=1),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, Nmodes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)

## Loading the instrument configuration

Each instrument has a dedicated params file (here `Rama_params.py`) holding the plain-dict configs consumed positionally by the pipeline constructors — `WFSParams`, `AtmosParams`, `LoopParams`, `TrainParams`, `DMParams`. There is no YAML/JSON config layer in this codebase; these dicts *are* the configuration mechanism.

In [ ]:
device = 'cuda' # set to "cpu" if Cuda is not available

paramfile = 'Rama_params.py'  # file of experimental parameters

# Config extraction
AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']
DMParams = Config.fromfile(paramfile)['DMParams']

## Dataset

`PhaseDataset` generates atmospheric phase screens on the fly from a von Kármán PSD. Setting `dataset.generateClosedLoop = True` selects the closed-loop (AO-residual) PSD instead of raw open-loop turbulence — see `Tutorials/basics/01_Dataset.ipynb` for exactly what that does and doesn't change.

In [ ]:
# Dataset creation
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

## Loading the calibrated WFS and DM

We load the WFS mask geometry and DM misregistration/influence functions fit in `CalibrateExampleRamaTwin.ipynb`, then freeze them (`.eval()` + `requires_grad_(False)`) — in this notebook only the reconstructor network is trained, so gradients should not flow into the optical model or DM geometry. Re-centering the influence functions (masking by the pupil, then subtracting each actuator's mean over the pupil) removes DC offsets outside the aperture so DM commands don't inject unphysical piston into the reconstructed phase.

`PATH` uses `../../Data/Rama/` (not `../Data/Rama/`) since this notebook now lives in `Tutorials/Rama/`, one level deeper than `Tutorials/` — the old single-`..` path is stale after that move and would silently point at a nonexistent folder.

In [ ]:
PATH = "../../Data/Rama/"

# WFS creation
wfs = PyramidWFS(WFSParams, device)
wfs.LoadCalibration(PATH + "RamaWFS.pth")
wfs.eval()
wfs.requires_grad_(False)

dm = DeformableMirror(WFSParams, DMParams, device)
dm.LoadCalibration(PATH + "RamaDM.pth")
dm.eval()
dm.requires_grad_(False)
dm.IF *= dataset.pupil
dm.IF[:, dataset.pupil] -= dm.IF[:, dataset.pupil].mean(dim=(-1), keepdim=True)

## Modes-to-commands matrix

`M2C` converts a vector of modal coefficients into DM actuator commands, so `dm(M2C.T)` gives the full-resolution phase produced by each individual mode on its own. `z_inv`, its pseudo-inverse, does the reverse: projecting a full-resolution phase screen onto modal coefficients. This is how the training loop obtains the "ground truth" modal coefficients for a given turbulence phase screen, to compare against the reconstructor's prediction.

In [ ]:
M2C = np.load(PATH + "M2C.npy")
M2C = torch.from_numpy(M2C).to(device=device, dtype=torch.float32)
M2C = M2C[:, :DMParams["Nmodes"]]  # keep only the modes this reconstructor is trained to output

z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))  # full-resolution phase -> modal coefficients

## Elongated pyramid mask

`wfs.BuildPrismMask(pupil_proportion, Nsamples)` turns the standard 4-facet pyramid into an *elongated* one — it samples `Nsamples` displaced copies of the mask across a `pupil_proportion`-wide range and stacks them, approximating a pyramid whose tip is smeared out (e.g. by chromatic dispersion or a deliberately extended edge) rather than a single sharp point. This directly overwrites `wfs.mask`, which everything downstream (`reference_intensity`, frame normalization) depends on.

**This must run before `FramePreprocess`/`ProcessReference` below, not after** — the notebook used to call it *after* building the frame preprocessor and its reference intensity, which were then computed from the pre-elongation (standard) mask and left stale relative to the mask actually used during training. Moved here, ahead of `FramePreprocess`, so the reference and normalization are computed from the same elongated mask the WFS actually propagates through.

In [ ]:
wfs.BuildPrismMask(14 / 34, 10)

## Frame preprocessing

`FramePreprocess` crops the individual pupil images out of the raw WFS detector frame and reference-subtracts/normalizes them before they reach the reconstructor. `ProcessReference` records the WFS's own flat-wavefront reference intensity, which is subtracted from every subsequent frame passed through `ProcessFrame`.

In [ ]:
# frame processor creation
framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

## Instantiating the reconstructor

We build the network and (if present) resume from a previously trained checkpoint further below, once the `Trainer` exists — see the "The Trainer" section.

In [ ]:
# Phase reconstructor
phaseReconstructor = PWFSNet(DMParams).to(device=device)

total_params = sum(p.numel() for p in phaseReconstructor.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

## Optimizer and loss

We optimize only the reconstructor's parameters, with AdamW. `LogResidualVarianceLoss` is a physics-aware loss: it computes `ln(var(residual phase over the pupil))`, i.e. the log-variance of the wavefront error in radians², which is directly tied to the residual RMS/Strehl ratio the AO loop would achieve — not just an abstract regression loss on the mode coefficients.

In [ ]:
# Optimization parameters (learning rate lr and nb of runs)
lrn = TrainParams['lrn']
num_iterations = 1 # Closed loop iterations

optimizer_n = torch.optim.AdamW(phaseReconstructor.parameters(), lrn, fused=True)

# Setting the loss
loss_variance = LogResidualVarianceLoss(dataset.pupil)

Re-run the cell below with a different `TrainRunNb`/`num_iterations` to control how many optimizer steps `trainer.train()` performs, without re-creating (and losing the momentum state of) the optimizer defined above.

In [ ]:
TrainRunNb = TrainParams['TrainRunNb']
num_iterations = 1

## The Trainer

`AI4AO.Trainer.Trainer` bundles the WFS, DM, frame preprocessor, modal basis (`M2C`), reconstructor, dataset, loss and optimizer, and implements the closed-loop training step (see `AI4AO/Trainer.py`). It also exposes `save_checkpoint`/`load_checkpoint` for persisting reconstructor + optimizer state, and the `evaluate`/`plot_losses` helpers used further down.

This notebook keeps the load/save split it always had: it resumes from `RamaCNN.pth` (the non-elongated baseline reconstructor) as a warm start, then saves the fine-tuned result to a separate `RamaCNNElongated.pth` — so fine-tuning for the elongated-pyramid configuration below never overwrites the original baseline checkpoint. Note: a checkpoint saved by the older `torch.save(phaseReconstructor.state_dict(), ...)` pattern is a raw state dict, not the wrapped format `save_checkpoint` writes, so loading it here will raise `KeyError` and fall back to training from scratch — re-save once with `trainer.save_checkpoint(...)` to make it loadable by `load_checkpoint` going forward.

In [ ]:
LOAD_CHECKPOINT_PATH = PATH + "RamaCNN.pth"
SAVE_CHECKPOINT_PATH = PATH + "RamaCNNElongated.pth"

trainer = Trainer(wfs=wfs,
                  framePreprocessor=framePreprocessor,
                  dm=dm,
                  M2C=M2C,
                  phaseReconstructor=phaseReconstructor,
                  dataset=dataset,
                  loss=loss_variance,
                  optimizer=optimizer_n)

try:
    trainer.load_checkpoint(LOAD_CHECKPOINT_PATH, load_optimizer=False)
except KeyError:
    # Existing checkpoint predates save_checkpoint's format (a raw state_dict);
    # once re-saved with trainer.save_checkpoint it will load cleanly here.
    print("Starting from scratch")

## Training

`trainer.train(training_steps, closed_loop_iterations)` runs `training_steps` closed-loop optimizer updates. `closed_loop_iterations` sets how many AO-loop steps are simulated — and backpropagated through — per optimizer update; with more than 1, the reconstructor is trained to perform well *given* its own previous corrections, rather than only on independent open-loop frames.

It returns two per-step loss trackers: `loss_tracker`, the network's actual training loss, and `loss_tracker_ideal`, the loss that would result from a perfect projection of the true residual phase onto the modal basis instead of the network's prediction — a lower bound to compare against.

In [ ]:
loss_tracker, loss_tracker_ideal = trainer.train(TrainRunNb, num_iterations)

`trainer.plot_losses` smooths and plots both trackers together. The gap between the training loss and the ideal-loss lower bound indicates how much reconstruction performance is still on the table for the network to gain, versus how much is fundamental to the chosen modal basis and WFS.

In [ ]:
trainer.plot_losses(loss_tracker, loss_tracker_ideal, ylim=(-3, 1))

## Saving

Persist the trained reconstructor, along with the optimizer state (for resuming later), to `SAVE_CHECKPOINT_PATH` — `RamaCNNElongated.pth`, not the `RamaCNN.pth` baseline we loaded from.

In [ ]:
trainer.save_checkpoint(SAVE_CHECKPOINT_PATH)

## Visualizing a closed loop

`trainer.evaluate()` runs a no-grad closed-loop rollout (reconstructor in `.eval()` mode, no pupil noise injected) and returns an `EvaluationResult` holding the phase, pupil, reconstructed phase, residual phase and WFS frames at every simulated step, ready to animate.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_frames = 60
result = trainer.evaluate(n_steps=n_frames, dataset=dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.phase[0], "title": "Input phase", "same_scale": True},
        {"tensor": result.residual_phase[0], "title": "Residual phase", "scale_reference": result.phase[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.phase[i], "title": "Input phase", "same_scale": True},
            {"tensor": result.residual_phase[i], "title": "Residual phase", "scale_reference": result.phase[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes, 
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())

### With scintillation

Same rollout, but on a fresh dataset with `AtmosParams['Scintillation'] = True` — amplitude fluctuations from angular-spectrum propagation are added on top of the phase screens, which the reconstructor was not trained to handle. Useful to see how it degrades outside its training distribution.

In [ ]:
scint_AtmosParams = AtmosParams.copy()
scint_AtmosParams['Scintillation'] = True
scint_dataset = PhaseDataset(WFSParams, scint_AtmosParams, LoopParams, DMParams, device)
scint_dataset.generateClosedLoop = True

result = trainer.evaluate(n_steps=n_frames, dataset=scint_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.pupil[0] * wfs.pupil, "title": "Pupil amplitude"},
        {"tensor": result.phase[0], "title": "Input phase", "same_scale": True},
        {"tensor": result.residual_phase[0], "title": "Residual phase", "scale_reference": result.phase[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    same_scale=True,
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.pupil[i] * wfs.pupil, "title": "Pupil amplitude"},
            {"tensor": result.phase[i], "title": "Input phase", "same_scale": True},
            {"tensor": result.residual_phase[i], "title": "Residual phase", "scale_reference": result.phase[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes, 
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())